# Heatmap

In [ ]:
import matplotlib.pyplot as plt
import warnings
import pandas as pd
import numpy as np
warnings.filterwarnings("ignore")

In [ ]:
%run ./peak_plot

In [ ]:
match_method = "calendar2_1year"

# =====================================================
# Load matching result
# =====================================================

matches = pd.read_parquet(
    f"/lakehouse/default/Files/output/matching_high/{match_method}/matches"
)

matches["cohort"] = pd.to_datetime(matches["adoption_month"]).dt.to_period("M")

treated_ids = matches["treated_id"].unique()
control_ids = matches["control_id"].unique()

print(matches.head())
print(f"Treated IDs: {len(treated_ids)}")
print(f"Matched control IDs: {len(control_ids)}")


# =====================================================
# Load full monthly data
# IMPORTANT: keep this as full sample
# =====================================================

month_result_full = pd.read_parquet("/lakehouse/default/Files/month_data")

print("Full month_result:", month_result_full.shape)
print("Full households:", month_result_full["aID"].nunique())


# =====================================================
# Construct matched sample mapping
# =====================================================

treated_map = matches[["treated_id", "cohort"]].drop_duplicates().copy()
treated_map.columns = ["aID", "cohort"]
treated_map["treatment"] = 1

control_map = matches[["control_id", "cohort"]].copy()
control_map.columns = ["aID", "cohort"]
control_map["treatment"] = 0

match_map = pd.concat([treated_map, control_map], axis=0)

print(match_map.head())


# =====================================================
# Matched sample only
# Use this for main matched analysis
# =====================================================

month_result_matched = month_result_full.merge(
    match_map,
    on="aID",
    how="inner"
)

print("Matched month_result:", month_result_matched.shape)
print("Matched households:", month_result_matched["aID"].nunique())


# =====================================================
# Identify adopter / never adopter groups in full sample
# =====================================================

all_adopter_ids = month_result_full.loc[
    month_result_full["tariff_start"].notna(), "aID"
].unique()

all_never_ids = month_result_full.loc[
    month_result_full["tariff_start"].isna(), "aID"
].unique()

matched_never_ids = control_ids

unmatched_never_ids = np.setdiff1d(
    all_never_ids,
    matched_never_ids
)

print(f"All adopters: {len(all_adopter_ids)}")
print(f"All never adopters: {len(all_never_ids)}")
print(f"Matched never adopters: {len(matched_never_ids)}")
print(f"Unmatched never adopters: {len(unmatched_never_ids)}")


In [ ]:
# Peak consumption heatmap: E(consumption | peak hour)
plot_tariff_consumption_heatmap(
    month_result_matched[month_result_matched["price"] == "high"],
    price_label="high"
)

plot_tariff_consumption_heatmap(
    month_result_matched[month_result_matched["price"] == "all"],
    price_label="all"
)

plot_tariff_consumption_heatmap(
    month_result_matched[month_result_matched["price"] == "low"],
    price_label="low"
)

1×2
Everyone overall consumption (Before tariff)
Everyone overall consumption (After tariff)

1×4
All never adopters
All adopters (Before tarff)
Matched never adopters
Unmatched never adopters

In [ ]:
df_high_full = month_result_full[month_result_full["price"] == "high"].copy()
df_low_full  = month_result_full[month_result_full["price"] == "low"].copy()
df_all_full  = month_result_full[month_result_full["price"] == "all"].copy()

common_vmax_population = get_population_heatmap_common_vmax(
    dfs=[df_high_full, df_low_full, df_all_full],
    matched_control_ids=matched_never_ids
)

for df, label in [
    (df_high_full, "high"),
    (df_low_full, "low"),
    (df_all_full, "all")
]:
    plot_all_households_before_after_heatmap(
        df,
        price_label=label,
        consumption_vmax=common_vmax_population
    )

    plot_matching_status_heatmaps(
        df,
        matched_control_ids=matched_never_ids,
        price_label=label,
        consumption_vmax=common_vmax_population
    )